# Chapitre 0 — Pourquoi Pandas ?

Ce chapitre ne présente encore aucune méthode avancée. Son objectif est de savoir **ce que l'on manipule et pourquoi**, avant d'apprendre à le manipuler :

1. le problème que Pandas résout ;
2. les situations où on l'utilise (et celles où on ne l'utilise pas) ;
3. sa place par rapport à Python, NumPy et SQL ;
4. ce qu'est un jeu de données ;
5. ce qu'est un DataFrame : lignes, colonnes, index, valeurs, types ;
6. ce qu'est une Series et son lien avec le DataFrame ;
7. pourquoi Pandas est central en data science et en IA.

## 1. Le problème : des données tabulaires, hétérogènes et imparfaites

Une boutique en ligne enregistre ses commandes. Chaque commande possède un identifiant, un client, un produit, une date, une quantité, un canal de vente et un statut.

| commande_id | client_id | produit_id | date_commande | quantite | canal | statut |
|---|---|---|---|---|---|---|
| 5001 | 999 | P08 | 2025-01-02 20:20 | 5 | web | livrée |
| 5002 | 105 | P09 | 2025-01-04 19:56 | 1 | web | livrée |
| 5003 | 998 | P12 | 2025-01-06 09:09 | 1 | web | annulée |

Trois caractéristiques rendent ces données difficiles à traiter avec les seuls outils vus jusqu'ici :

- **elles sont hétérogènes** : des entiers, des textes, des dates, des décimaux cohabitent dans la même table ;
- **elles sont étiquetées** : on veut parler de « la colonne `quantite` » ou « de la commande 5002 », pas de « la colonne 4 » ou « de la ligne 1 » ;
- **elles sont imparfaites** : des valeurs manquent, des doublons existent, des textes sont mal saisis.

Les questions que l'on se pose sont pourtant simples : combien de commandes par canal ? Quel chiffre d'affaires par mois ? Quels clients n'ont jamais commandé ? Pandas existe pour répondre à ces questions en quelques lignes, de façon lisible et rapide.

### Avec Python seul

Commençons avec ce que vous connaissez : une liste de dictionnaires. Calculons la quantité moyenne commandée par canal.

In [1]:
commandes = [
    {"commande_id": 5001, "canal": "web",     "quantite": 1},
    {"commande_id": 5002, "canal": "magasin", "quantite": 2},
    {"commande_id": 5003, "canal": "web",     "quantite": 1},
    {"commande_id": 5004, "canal": "web",     "quantite": 3},
    {"commande_id": 5005, "canal": "magasin", "quantite": None},   # valeur manquante
]

totaux, effectifs = {}, {}
for c in commandes:
    if c["quantite"] is None:          # il faut penser à gérer le cas manquant
        continue
    totaux[c["canal"]] = totaux.get(c["canal"], 0) + c["quantite"]
    effectifs[c["canal"]] = effectifs.get(c["canal"], 0) + 1

{canal: totaux[canal] / effectifs[canal] for canal in totaux}

{'web': 1.6666666666666667, 'magasin': 2.0}

Le résultat est correct, mais le code mélange trois préoccupations : le parcours des lignes, la gestion des valeurs manquantes et le calcul lui-même. Chaque nouvelle question demande une nouvelle boucle.

### Avec Pandas

In [5]:
import pandas as pd

df = pd.DataFrame(commandes)
print(df.groupby("canal")["quantite"].mean())
print('***')
print(df)

canal
magasin    2.000000
web        1.666667
Name: quantite, dtype: float64
***
   commande_id    canal  quantite
0         5001      web       1.0
1         5002  magasin       2.0
2         5003      web       1.0
3         5004      web       3.0
4         5005  magasin       NaN


Une ligne exprime l'intention : *grouper par canal, prendre la quantité, calculer la moyenne*. La valeur manquante est ignorée automatiquement par `mean()`. Ce style **déclaratif** est très proche de SQL :

```sql
SELECT canal, AVG(quantite) FROM commandes GROUP BY canal;
```

## 2. Dans quelles situations utilise-t-on Pandas ?

Pandas est l'outil de référence en Python dès que les données tiennent en mémoire et prennent la forme d'un tableau.

| Situation | Exemple |
|---|---|
| Explorer un fichier inconnu | ouvrir un CSV de 270 000 lignes et comprendre son contenu en quelques commandes |
| Nettoyer des données | corriger des types, des doublons, des textes mal saisis, des valeurs manquantes |
| Analyser | statistiques, regroupements, tableaux croisés |
| Assembler plusieurs sources | joindre un export Excel, une table SQL et une réponse d'API |
| Préparer des données pour un modèle | encoder des catégories, créer des variables, séparer les colonnes cibles |
| Produire un résultat | exporter un CSV, un fichier Excel, une table SQL |

Pandas n'est **pas** l'outil adapté pour :

- stocker des données de façon durable et partagée : c'est le rôle d'une base de données ;
- du calcul numérique pur sur de grands tableaux homogènes (images, matrices) : NumPy suffit et va plus vite ;
- des volumes qui dépassent largement la mémoire vive : on lit alors par morceaux (approfondissement A2) ou on utilise d'autres outils (Polars, DuckDB, Spark).

## 3. Pandas, Python, NumPy et SQL

Pandas ne remplace aucun de ces outils : il s'appuie sur eux.

| | Python (listes, dict) | NumPy | SQL | Pandas |
|---|---|---|---|---|
| Structure principale | listes, dictionnaires | `ndarray` (tableau homogène) | table dans une base | `DataFrame` et `Series` |
| Types | mélange libre | un seul type par tableau | un type par colonne | un type par colonne |
| Accès | par position ou clé | par position | par nom de colonne, `WHERE` | par étiquette **et** par position |
| Calculs | boucles | vectorisés | requêtes déclaratives | vectorisés et déclaratifs |
| Valeurs manquantes | `None`, à gérer soi-même | `nan` (flottants uniquement) | `NULL` | `NaN`, `NaT`, `<NA>`, gérées par la plupart des méthodes |
| Où vivent les données | mémoire | mémoire | disque (serveur) | mémoire |

Deux idées à retenir :

- **Chaque colonne d'un DataFrame est stockée comme un tableau NumPy** (ou un tableau équivalent). Tout ce que vous savez sur la vectorisation et les `dtype` s'applique.
- **Pandas emprunte à SQL sa logique de table** : sélectionner, filtrer, grouper, joindre. Les mêmes questions s'écrivent avec des méthodes Python au lieu de requêtes.

### Pourquoi pas NumPy seul ?

Un tableau NumPy n'a qu'un seul type. Mélanger texte et nombres force une conversion de tout le tableau.

In [6]:
import numpy as np

tableau = np.array([
    [5001, "web", 1],
    [5002, "magasin", 2],
])
tableau, tableau.dtype

(array([['5001', 'web', '1'],
        ['5002', 'magasin', '2']], dtype='<U21'),
 dtype('<U21'))

Le type affiché est `<U21` : une chaîne de caractères Unicode. Les nombres 5001 et 1 sont devenus du texte, si bien que `tableau[:, 2].sum()` ne fonctionne plus. NumPy a choisi le seul type capable de représenter toutes les valeurs.

Un DataFrame contourne ce problème en stockant **chaque colonne séparément**, avec son propre type.

In [7]:
pd.DataFrame({"commande_id": [5001, 5002], "canal": ["web", "magasin"], "quantite": [1, 2]}).dtypes

commande_id    int64
canal            str
quantite       int64
dtype: object

## 4. Qu'est-ce qu'un jeu de données ?

Un **jeu de données** (dataset) est une collection d'observations décrites par les mêmes caractéristiques.

- Une **observation** (ou enregistrement, individu, ligne) : une commande, un athlète, une mesure de capteur.
- Une **variable** (ou caractéristique, attribut, colonne) : la quantité, le nom, la température.

Un jeu de données est dit **bien rangé** (*tidy*) lorsque :

1. chaque variable forme une colonne ;
2. chaque observation forme une ligne ;
3. chaque cellule contient une seule valeur.

Ce format est celui qu'attendent Pandas, les bases SQL et les bibliothèques de machine learning. Une partie importante du travail consiste à amener les données dans cette forme (chapitres 5 et 9).

Les variables se classent en grandes familles, qui déterminent les opérations possibles :

| Famille | Exemples | Opérations naturelles |
|---|---|---|
| Numérique continue | prix, température | moyenne, écart-type, somme |
| Numérique discrète | quantité, âge | comptage, moyenne |
| Catégorielle | canal, segment, ville | comptage par modalité, regroupement |
| Texte libre | nom, description | recherche, nettoyage |
| Date et heure | date de commande | extraction du mois, différence de dates |
| Booléenne | client actif oui/non | filtrage, proportion |

## 5. Anatomie d'un DataFrame

Chargeons le fichier des clients de la boutique.

In [8]:
clients = pd.read_csv("../donnees/demo/clients.csv")
clients.head()

,client_id,prenom,ville,segment,age,date_inscription
0,101,Amina,Anvers,Particulier,40.0,2025-02-09
1,102,Lucas,Liège,Pro,51.0,2023-09-11
2,103,Sofia,Charleroi,Particulier,51.0,2025-04-05
3,104,Noah,Anvers,Particulier,NaN,2023-09-22
4,105,Lina,Bruxelles,Particulier,60.0,2024-01-26


Un DataFrame se décompose en quatre éléments :

```
                    colonnes (columns) : des noms
                 ┌──────────┬───────────┬────────┐
                 │ prenom   │ ville     │ age    │
         ┌───────┼──────────┼───────────┼────────┤
         │   0   │ Amina    │ Anvers    │ 40.0   │  ← une ligne = une observation
 index → │   1   │ Lucas    │ Liège     │ 51.0   │
         │   2   │ Sofia    │ Charleroi │ 51.0   │
         │   3   │ Noah     │ Anvers    │ NaN    │  ← valeur manquante
         └───────┴──────────┴───────────┴────────┘
                   str        str         float64   ← un type par colonne (dtypes)
```

- **Les colonnes** portent un nom. Chaque colonne a un seul type.
- **L'index** étiquette les lignes. Par défaut, c'est une suite d'entiers 0, 1, 2… mais il peut contenir des identifiants, des dates, des textes.
- **Les valeurs** sont le contenu des cellules.
- **Les types** (`dtypes`) décrivent la nature de chaque colonne.

In [9]:
clients.columns

Index(['client_id', 'prenom', 'ville', 'segment', 'age', 'date_inscription'],
      dtype='str')

In [10]:
clients.index

RangeIndex(start=0, stop=30, step=1)

In [11]:
clients.dtypes

client_id             int64
prenom                  str
ville                   str
segment                 str
age                 float64
date_inscription        str
dtype: object

Lecture des types :

- `int64` : entiers sur 64 bits ;
- `float64` : décimaux. La colonne `age` est en `float64` alors que les âges sont entiers : la colonne contient des valeurs manquantes, et le `NaN` de NumPy est un flottant ;
- `str` : texte. C'est le type par défaut du texte depuis Pandas 3. Vous rencontrerez aussi `object` dans du code plus ancien.

La date d'inscription est lue comme du texte : un CSV ne contient que des caractères, et Pandas ne devine pas les dates sans qu'on le lui demande (chapitre 3).

In [12]:
clients.shape    # (nombre de lignes, nombre de colonnes)

(30, 6)

> **Prédiction.** Dans la cellule `clients.head()`, l'âge de la ligne 3 apparaît comme `NaN`. Combien de valeurs manquantes contient la colonne `age` ? Proposez une méthode avant d'exécuter la cellule suivante.

In [23]:
clients["age"].isna().sum()

np.int64(3)

## 6. La Series : une colonne étiquetée

Quand on extrait une colonne d'un DataFrame, on obtient une **Series**.

In [24]:
villes = clients["ville"]
type(villes)

pandas.Series

In [33]:
villes.head()

0       Anvers
1        Liège
2    Charleroi
3       Anvers
4    Bruxelles
Name: ville, dtype: str

Une Series, c'est **un tableau à une dimension, un type, et un index**. Elle conserve l'index du DataFrame dont elle provient.

On peut voir un DataFrame de deux façons équivalentes :

- un **dictionnaire de Series** qui partagent le même index (une Series par colonne) ;
- un **tableau à deux dimensions** dont les lignes et les colonnes sont étiquetées.

In [36]:
serie_age = pd.Series([40, 51, 51], index=["Amina", "Lucas", "Sofia"], name="age")
serie_age

Amina    40
Lucas    51
Sofia    51
Name: age, dtype: int64

In [44]:
serie_ville = pd.Series(["Anvers", "Liège", "Charleroi", "Chatelet"], index=["Amina", "Fabian", "Sofia", "Anis"], name="ville")
pd.DataFrame({"age": serie_age, "ville": serie_ville})

,age,ville
Amina,40.0,Anvers
Anis,NaN,Chatelet
Fabian,NaN,Liège
Lucas,51.0,NaN
Sofia,51.0,Charleroi


Les deux Series partagent le même index : Pandas les aligne ligne à ligne par étiquette. Cette notion d'**alignement par l'index** est au cœur de Pandas. Le chapitre 1 lui est consacré.

## 7. Pourquoi Pandas est central en data science et en IA

Un modèle de machine learning n'apprend que ce qu'on lui donne. Dans un projet réel, la majeure partie du travail se situe **avant** l'entraînement :

```
 sources brutes        exploration        nettoyage          préparation         modèle
 CSV, SQL, API   →   info, describe  →  types, NaN,    →   encodage,       →   scikit-learn,
 Excel, JSON          value_counts        doublons           jointures,           PyTorch…
                                                             variables
 └──────────────────────────── Pandas ──────────────────────────────┘
```

Pandas intervient à chaque étape avant le modèle :

- **charger** des données de sources variées dans une structure unique ;
- **comprendre** leur contenu (distributions, valeurs manquantes, anomalies) ;
- **nettoyer** sans perdre la trace de ce que l'on a modifié ;
- **construire des variables** (*features*) : agrégats par client, indicateurs temporels, encodage des catégories ;
- **produire** une matrice numérique (`df.to_numpy()`) que les bibliothèques de modélisation consomment directement.

La plupart des bibliothèques de l'écosystème (scikit-learn, seaborn, statsmodels, Hugging Face Datasets) acceptent ou produisent des DataFrame.

## 8. L'environnement du cours

Le projet se lance avec `uv sync` à la racine, puis `uv run jupyter lab`. Vérifions les versions utilisées.

In [ ]:
import sys
print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("NumPy  :", np.__version__)

Ce cours utilise **Pandas 3**. Beaucoup de tutoriels en ligne et une partie du code que vous rencontrerez datent de Pandas 1 ou 2. Les différences importantes seront signalées au fil des chapitres, notamment :

- le texte est stocké dans un type `str` dédié, et non plus `object` ;
- le mode *Copy-on-Write* : une modification passe toujours par l'objet lui-même (`df["col"] = ...`), jamais par un intermédiaire (chapitre 5) ;
- plusieurs méthodes ont disparu : `DataFrame.append`, `applymap`, et certains alias de fréquence comme `"M"`.

Deux références à garder ouvertes :

- la documentation officielle : <https://pandas.pydata.org/docs/>, avec l'aide intégrée (`pd.read_csv?` dans un notebook) ;
- la page de comparaison avec SQL : <https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html>.

## À retenir

- Pandas sert à manipuler des **données tabulaires, hétérogènes et étiquetées**, qui tiennent en mémoire.
- Un **DataFrame** est un tableau dont les colonnes ont un nom et un type, et dont les lignes sont étiquetées par un **index**.
- Une **Series** est une colonne : un tableau à une dimension, un type, un index.
- Chaque colonne repose sur un tableau de type NumPy : les calculs sont vectorisés.
- La logique des opérations (filtrer, grouper, joindre) est celle de SQL, exprimée en Python.
- En IA, Pandas couvre tout le chemin entre les données brutes et la matrice qui alimente un modèle.